# Custom FIN-RoBERTa Model Evaluation

This notebook evaluates the custom FIN-RoBERTa model on the Financial PhraseBank dataset.

- **Model:** [alasteirho/FIN-RoBERTa-Custom](https://huggingface.co/alasteirho/FIN-RoBERTa-Custom)
- **Evaluation Dataset:** [takala/financial_phrasebank](https://huggingface.co/datasets/takala/financial_phrasebank) (sentences_allagree - 100% annotator agreement)
- **Labels:** negative, neutral, positive

In [4]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from datasets import load_dataset
import os

# ======== Configuration ========
model_name = "alasteirho/FIN-RoBERTa-Custom"

# ======== Load Model ========
print("Loading Fin-RoBERTa model and tokenizer")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model = model.to(device)
model.eval()
print(f"Model loaded on {device}")

Loading Fin-RoBERTa model and tokenizer
Model loaded on cuda


In [5]:
# ======== Inference Function ========
def classify_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    pred_idx = torch.argmax(probs, dim=-1).item()
    # Fin-RoBERTa labels: 0=negative, 1=neutral, 2=positive
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    return label_map[pred_idx]

def evaluate_dataset(df, dataset_name):
    print(f"\n{'='*50}")
    print(f"Evaluating on: {dataset_name}")
    print(f"{'='*50}")
    print(f"Loaded {len(df)} samples")

    # Run predictions
    preds = []
    for text in tqdm(df["sentence"], desc="Predicting"):
        preds.append(classify_text(str(text)))

    # Compute metrics
    y_true = df["sentiment"].astype(str).str.lower()
    y_pred = preds

    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, digits=3)
    cm = confusion_matrix(y_true, y_pred, labels=["positive", "neutral", "negative"])

    print(f"\nAccuracy: {acc:.4f}")
    print(report)
    print("Confusion Matrix (rows=true, cols=pred):")
    print("Labels: [positive, neutral, negative]")
    print(cm)

    return acc

## Evaluate on Financial PhraseBank (100% All Agree)

In [6]:
# Load Financial PhraseBank dataset (100% all agree)
print("Loading Financial PhraseBank dataset from Hugging Face...")
dataset = load_dataset("takala/financial_phrasebank", "sentences_allagree", trust_remote_code=True)

# Convert to DataFrame
df_agree = pd.DataFrame(dataset["train"])

# Map label integers to sentiment strings
# Dataset labels: 0=negative, 1=neutral, 2=positive
label_map = {0: "negative", 1: "neutral", 2: "positive"}
df_agree["sentiment"] = df_agree["label"].map(label_map)

print(f"Dataset loaded: {len(df_agree)} samples")
print(df_agree["sentiment"].value_counts())

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'takala/financial_phrasebank' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading Financial PhraseBank dataset from Hugging Face...


Using the latest cached version of the dataset since takala/financial_phrasebank couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'sentences_allagree' at C:\Users\Alasteir\.cache\huggingface\datasets\takala___financial_phrasebank\sentences_allagree\0.0.0\0dd3028d70cbd18ded8887e65e83343b03a50482 (last modified on Sat Jan 24 20:38:23 2026).


Dataset loaded: 2264 samples
sentiment
neutral     1391
positive     570
negative     303
Name: count, dtype: int64


In [7]:
# Evaluate on Financial PhraseBank
acc_agree = evaluate_dataset(df_agree, "Financial PhraseBank (sentences_allagree)")


Evaluating on: Financial PhraseBank (sentences_allagree)
Loaded 2264 samples


Predicting: 100%|██████████| 2264/2264 [00:11<00:00, 189.43it/s]


Accuracy: 0.9929
              precision    recall  f1-score   support

    negative      0.987     0.993     0.990       303
     neutral      0.995     0.996     0.996      1391
    positive      0.991     0.984     0.988       570

    accuracy                          0.993      2264
   macro avg      0.991     0.991     0.991      2264
weighted avg      0.993     0.993     0.993      2264

Confusion Matrix (rows=true, cols=pred):
Labels: [positive, neutral, negative]
[[ 561    5    4]
 [   5 1386    0]
 [   0    2  301]]
